In [ ]:
!pip install sentencepiece

In [ ]:
import pandas as pd
import sentencepiece as spm
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

os.listdir('/content/drive/MyDrive/Nepali_Dataset')

['small.csv', 'medium.csv', 'large.csv', 'processed']

In [ ]:
small = pd.read_csv(
    "/content/drive/MyDrive/Nepali_Dataset/small.csv"
)

medium = pd.read_csv(
    "/content/drive/MyDrive/Nepali_Dataset/medium.csv"
)

large = pd.read_csv(
    "/content/drive/MyDrive/Nepali_Dataset/large.csv"
)

In [ ]:
print("Small:", small.shape)
print("Medium:", medium.shape)
print("Large:", large.shape)

print("\nColumns:", small.columns.tolist())


Small: (17241, 3)
Medium: (21885, 3)
Large: (11403, 3)

Columns: ['id', 'content', 'summary']


In [ ]:
for name, df in [
    ("Small", small),
    ("Medium", medium),
    ("Large", large)
]:
    print(f"\n{name} Dataset")
    print("Missing content:", df["content"].isna().sum())
    print("Missing summary:", df["summary"].isna().sum())
    print("Duplicate IDs:", df["id"].duplicated().sum())


Small Dataset
Missing content: 0
Missing summary: 0
Duplicate IDs: 0

Medium Dataset
Missing content: 0
Missing summary: 0
Duplicate IDs: 0

Large Dataset
Missing content: 0
Missing summary: 0
Duplicate IDs: 0


In [ ]:
df = pd.concat(
    [small, medium, large],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)
print(df.head(3))

Combined dataset shape: (50529, 3)
    id                                            content  \
0  185  लक्ष्मी इन्टरकन्टिनेन्टल प्रा. लि.ले हुन्डाई इ...   
1  189  बेइजिङ। चीनले अमेरिका, युरोपेली संघ (इयु), ताइ...   
2  190  काठमाडौं । सम्पत्ति शुद्धीकरण सम्बन्धी कानुनको...   

                                             summary  
0  लक्ष्मी इन्टरकन्टिनेन्टलले हुन्डाईका इलेक्ट्रि...  
1  चीनले अमेरिका, युरोपेली संघ, ताइवान र जापानबाट...  
2  सम्पत्ति शुद्धीकरण सम्बन्धी निर्देशन पालना नगर...  


In [ ]:
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Duplicate articles:", df["content"].duplicated().sum())
print("Duplicate summaries:", df["summary"].duplicated().sum())
print("Duplicate article-summary pairs:", df.duplicated(subset=["content", "summary"]).sum())

Duplicate IDs: 0
Duplicate articles: 615
Duplicate summaries: 132
Duplicate article-summary pairs: 0


In [ ]:
duplicate_articles = df[df["content"].duplicated(keep=False)]

print("Rows containing repeated articles:", len(duplicate_articles))
display(duplicate_articles[["id", "content", "summary"]].head(6))

Rows containing repeated articles: 1228


,id,content,summary
0,185,लक्ष्मी इन्टरकन्टिनेन्टल प्रा. लि.ले हुन्डाई इ...,लक्ष्मी इन्टरकन्टिनेन्टलले हुन्डाईका इलेक्ट्रि...
2,190,काठमाडौं । सम्पत्ति शुद्धीकरण सम्बन्धी कानुनको...,सम्पत्ति शुद्धीकरण सम्बन्धी निर्देशन पालना नगर...
14,224,काठमाडौं । पिपुल्स पावर कम्पनीले हकप्रद सेयर न...,पिपुल्स पावर कम्पनीलाई चुक्ता पुँजीको ५० प्रति...
25,241,काठमाडौं। होटलहरुको वित्तीय अवस्थामा सुधार आएक...,नेप्सेमा सूचीकृत ६ होटलले चालु आर्थिक वर्षको ९...
35,265,काठमाडौं । नेपाल टेलिकमले लाङटाङ-क्यान्जिन पदम...,नेपाल टेलिकमले लाङटाङ-क्यान्जिन पदमार्गमा पर्य...
56,307,काठमाडौं। आन्तरिक अर्थतन्त्र शिथिल हुँदा विदेश...,नेपालको आन्तरिक अर्थतन्त्र शिथिल भए तापनि विदे...


In [ ]:
repeated = (
    df[df["content"].duplicated(keep=False)]
    .groupby("content")
    .agg(
        occurrences=("id", "count"),
        unique_summaries=("summary", "nunique")
    )
    .reset_index()
)

print("Repeated article groups:", len(repeated))
print("Groups with different summaries:",
      (repeated["unique_summaries"] > 1).sum())
print("\nSummary variation:")
print(repeated["unique_summaries"].value_counts().sort_index())

Repeated article groups: 613
Groups with different summaries: 613

Summary variation:
unique_summaries
2    611
3      2
Name: count, dtype: int64


In [ ]:
sample_article = repeated.iloc[0]["content"]

display(
    df[df["content"] == sample_article][["id", "summary"]]
)


,id,summary
44617,22542,"नेपाली संस्कृतिमा अभिभावकत्वको भूमिका गहिरो छ,..."
44619,22552,नेपाली संस्कृतिमा अभिभावकत्वको भूमिका महत्त्वप...


In [ ]:
df = df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

print("Dataset after removing repeated articles:", df.shape)

Dataset after removing repeated articles: (49914, 3)


In [ ]:
print("Empty articles:", (df["content"].str.strip() == "").sum())
print("Empty summaries:", (df["summary"].str.strip() == "").sum())

print("Article data type:", df["content"].dtype)
print("Summary data type:", df["summary"].dtype)

Empty articles: 0
Empty summaries: 0
Article data type: object
Summary data type: object


In [ ]:
df["content"] = df["content"].str.strip()
df["summary"] = df["summary"].str.strip()

print("Final dataset size:", len(df))

Final dataset size: 49914


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 39931
Validation: 4991
Testing: 4992


In [ ]:
all_text = []

for text in train_df["content"].astype(str):
    all_text.append(text)

for text in train_df["summary"].astype(str):
    all_text.append(text)

print("Total training texts:", len(all_text))

Total training texts: 79862


In [ ]:
corpus_file = "corpus.txt"

with open(
    corpus_file,
    "w",
    encoding="utf-8"
) as f:

    for line in all_text:
        f.write(line.strip())
        f.write("\n")

In [ ]:
with open(
    corpus_file,
    "r",
    encoding="utf-8"
) as f:

    for i in range(5):
        print(f.readline())

२८ पुस, बुटवल । सत्तारुढ दल कांग्रेसकै नेताले अध्यादेशबाट देश चलाउन खोजेको भन्दै सरकारको आलोचना गरेका छन् । कांग्रेस केन्द्रीय सदस्य तथा पूर्वपरराष्ट्रमन्त्री एनपी साउदले ले संसद्को नियमित अधिवेशन छलेर अध्यादेशको सहारामा देश चाउन खोज्नु गलत भएको भन्दै सरकारको आलोचना गरेका हुन् । सरकारसँग पर्याप्त बहुमत हुँदा पनि बैठक आह्वान नगरी संसद्लाई बन्धक बनाएर अध्यादेशमार्फत कानुन बनाउँदा चौतर्फी प्रश्न उठेको उनको भनाइ छ । बुटवलमा आइतबार पत्रकारहरूसँग बोल्दै नेता साउदले तत्काल संसद्को अधिवेशन बोलाउन पनि माग गरे । अध्यादेशमार्फत ल्याइएका विधेयक सर्वस्वीकार्य नहुने औँल्याए । ‘संसद्लाई लामो समयसम्म बन्धक बनाउँदै अध्ययादेशमार्फत अहिलेको सरकारले विभिन्न ऐन बनाइरहेको छ, जुन संविधानसम्मत छैन यस विषयमा चौतर्फी विरोध भइरहेका छन् यसले राम्रो सन्देश गइरहेको छैन,’ उनले भने । ठूला दलहरू मिलेर सरकार गठन हुँदासमेत स्थिरताको ग्यारेन्टी गर्न नसकिएको भन्दै नेता साउदले प्रधानमन्त्री केपी शर्मा ओलीले आफ्ना अभिव्यक्तिमा संवेदनशील हुनुपर्ने धारणा राखे । ‘प्रधानमन्त्री केपी ओलीले आफ्ना अभिव्यक्ति राख्दा संवेदनशील हुन ज

In [ ]:
VOCAB_SIZE = 32000

# Calculate maximum sentence length in UTF-8 bytes
max_line_length = 0

with open("corpus.txt", "r", encoding="utf-8") as f:
    for line in f:
        length = len(line.strip().encode("utf-8"))
        max_line_length = max(max_line_length, length)

safe_max_length = max(max_line_length + 1000, 10000)

print(f"Maximum sentence length (bytes): {max_line_length}")
print(f"Using max_sentence_length: {safe_max_length}")

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="nepali_sp",
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=1.0,
    max_sentence_length=safe_max_length,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

Maximum sentence length (bytes): 84501
Using max_sentence_length: 85501


True

In [ ]:
import os

print("Model exists:", os.path.exists("nepali_sp.model"))
print("Vocabulary exists:", os.path.exists("nepali_sp.vocab"))

Model exists: True
Vocabulary exists: True


In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("nepali_sp.model")

print("Vocabulary size:", sp.get_piece_size())
print("PAD ID:", sp.pad_id())
print("UNK ID:", sp.unk_id())
print("BOS ID:", sp.bos_id())
print("EOS ID:", sp.eos_id())

Vocabulary size: 32000
PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3


In [ ]:
sample_article = train_df.iloc[0]["content"]
sample_summary = train_df.iloc[0]["summary"]

print("Original summary:")
print(sample_summary)

encoded = sp.encode(sample_summary)

print("\nEncoded token IDs:")
print(encoded)

print("\nDecoded summary:")
print(sp.decode(encoded))

Original summary:
कांग्रेस नेता एनपी साउदले संसद अधिवेशन छलेर अध्यादेशबाट देश चलाउन खोजेको भन्दै सरकारको आलोचना गरेका छन्। उनले संसदलाई बन्धक बनाएर अध्यादेशमार्फत कानुन बनाउनु संविधान सम्मत नभएको बताए। साथै, उनले प्रधानमन्त्री ओलीलाई अभिव्यक्तिमा संवेदनशील हुन र दुई दलबीचको सहमति कार्यान्वयन गर्न सुझाव दिएका छन्।

Encoded token IDs:
[256, 206, 8124, 6019, 889, 978, 10468, 11481, 569, 1978, 1506, 75, 194, 1350, 33, 12, 8, 17, 7376, 4682, 626, 7833, 371, 2986, 467, 965, 650, 251, 27, 8, 169, 5, 17, 74, 2781, 1639, 10, 3177, 73, 7, 54, 9783, 9, 337, 314, 20, 386, 186, 12, 8]

Decoded summary:
कांग्रेस नेता एनपी साउदले संसद अधिवेशन छलेर अध्यादेशबाट देश चलाउन खोजेको भन्दै सरकारको आलोचना गरेका छन्। उनले संसदलाई बन्धक बनाएर अध्यादेशमार्फत कानुन बनाउनु संविधान सम्मत नभएको बताए। साथै, उनले प्रधानमन्त्री ओलीलाई अभिव्यक्तिमा संवेदनशील हुन र दुई दलबीचको सहमति कार्यान्वयन गर्न सुझाव दिएका छन्।


In [ ]:
article_lengths = [
    len(sp.encode(text))
    for text in train_df["content"]
]

summary_lengths = [
    len(sp.encode(text))
    for text in train_df["summary"]
]

import numpy as np

print("Article token lengths:")
print("Mean:", np.mean(article_lengths))
print("95th percentile:", np.percentile(article_lengths, 95))
print("Maximum:", np.max(article_lengths))

print("\nSummary token lengths:")
print("Mean:", np.mean(summary_lengths))
print("95th percentile:", np.percentile(summary_lengths, 95))
print("Maximum:", np.max(summary_lengths))

Article token lengths:
Mean: 353.69855500738777
95th percentile: 1038.0
Maximum: 5934

Summary token lengths:
Mean: 52.19205629711252
95th percentile: 77.0
Maximum: 143


In [ ]:
print(
    "Articles covered:",
    np.mean(np.array(article_lengths) <= 1100) * 100,
    "%"
)

print(
    "Summaries covered:",
    np.mean(np.array(summary_lengths) <= 100) * 100,
    "%"
)

Articles covered: 95.74766472164484 %
Summaries covered: 99.69447296586613 %


In [ ]:
sample = """
नेपाल सरकारले नयाँ आर्थिक नीति सार्वजनिक गरेको छ।
"""

tokens = sp.encode(
    sample,
    out_type=str
)

print(tokens)

['▁नेपाल', '▁सरकारले', '▁नयाँ', '▁आर्थिक', '▁नीति', '▁सार्वजनिक', '▁गरेको', '▁छ', '।']


In [ ]:
ids = sp.encode(
    sample,
    out_type=int
)

print(ids)

[35, 51, 104, 61, 212, 134, 16, 6, 8]


In [ ]:
decoded = sp.decode(ids)

print(decoded)

नेपाल सरकारले नयाँ आर्थिक नीति सार्वजनिक गरेको छ।


In [ ]:
import shutil
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/Nepali_Dataset/tokenizer")
output_dir.mkdir(parents=True, exist_ok=True)

shutil.copy("nepali_sp.model", output_dir / "nepali_sp.model")
shutil.copy("nepali_sp.vocab", output_dir / "nepali_sp.vocab")

print("Tokenizer files saved successfully!")
print(list(output_dir.iterdir()))

Tokenizer files saved successfully!
[PosixPath('/content/drive/MyDrive/Nepali_Dataset/tokenizer/nepali_sp.model'), PosixPath('/content/drive/MyDrive/Nepali_Dataset/tokenizer/nepali_sp.vocab')]


In [ ]:
output_dir = "/content/drive/MyDrive/Nepali_Dataset/processed"
Path(output_dir).mkdir(parents=True, exist_ok=True)

df.to_csv(f"{output_dir}/cleaned_dataset.csv", index=False)
train_df.to_csv(f"{output_dir}/train.csv", index=False)
val_df.to_csv(f"{output_dir}/validation.csv", index=False)
test_df.to_csv(f"{output_dir}/test.csv", index=False)

print("Datasets saved successfully!")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Datasets saved successfully!
Train: 39931
Validation: 4991
Test: 4992
